# Phase 1: Data Loading & Inspection
This phase covers:
- Creating a Spark session
- Loading the dataset with `sc.textFile()`
- Removing the header row
- Splitting rows into columns
- Displaying sample records
- Counting total records and partitions

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

Spark session created successfully.
Spark version: 4.1.1


In [ ]:
# sc.textFile() reads the file from the local filesystem or HDFS
raw_rdd = sc.textFile("Salary.csv")

print(f"Raw RDD loaded. Type: {type(raw_rdd)}")
print(f"First line (header): {raw_rdd.first()}")

Raw RDD loaded. Type: <class 'pyspark.core.rdd.RDD'>


First line (header): Rating,Company Name,Job Title,Salary,Salaries Reported,Location,Employment Status,Job Roles


In [ ]:
# Capture the header line so we can exclude it from the data
header = raw_rdd.first()

# Use filter() to remove the header row from the RDD
# Every row that is NOT equal to the header is kept
data_rdd = raw_rdd.filter(lambda row: row != header)

print(f"Header removed: '{header}'")
print(f"Remaining rows (first 5 after header removal):")
for row in data_rdd.take(5):
    print(" ", row)

Header removed: 'Rating,Company Name,Job Title,Salary,Salaries Reported,Location,Employment Status,Job Roles'
Remaining rows (first 2 after header removal):
  3.8,Sasken,Android Developer,400000,3,Bangalore,Full Time,Android
  4.5,Advanced Millennium Technologies,Android Developer,400000,3,Bangalore,Full Time,Android


In [5]:
# Use map() to split each CSV row string on commas
# This produces an RDD of lists, where each list is one record
split_rdd = data_rdd.map(lambda row: row.split(","))

print("Rows split into columns. Example record (list of fields):")
print(split_rdd.first())

Rows split into columns. Example record (list of fields):
['3.8', 'Sasken', 'Android Developer', '400000', '3', 'Bangalore', 'Full Time', 'Android']


In [6]:
# take(5) retrieves the first 5 records as a Python list (driver-side action)
# Useful for quick inspection without loading the entire dataset
sample = split_rdd.take(5)

print("Sample records (first 5):")
for i, record in enumerate(sample, 1):
    print(f"  [{i}] {record}")

Sample records (first 5):
  [1] ['3.8', 'Sasken', 'Android Developer', '400000', '3', 'Bangalore', 'Full Time', 'Android']
  [2] ['4.5', 'Advanced Millennium Technologies', 'Android Developer', '400000', '3', 'Bangalore', 'Full Time', 'Android']
  [3] ['4', 'Unacademy', 'Android Developer', '1000000', '3', 'Bangalore', 'Full Time', 'Android']
  [4] ['3.8', 'SnapBizz Cloudtech', 'Android Developer', '300000', '3', 'Bangalore', 'Full Time', 'Android']
  [5] ['4.4', 'Appoids Tech Solutions', 'Android Developer', '600000', '3', 'Bangalore', 'Full Time', 'Android']


In [7]:
# count() is an action that triggers computation and returns the total number of rows
total_records = split_rdd.count()

# getNumPartitions() returns the number of partitions the RDD is split into
num_partitions = split_rdd.getNumPartitions()

print(f"Total records in dataset : {total_records}")
print(f"Number of RDD partitions : {num_partitions}")

Total records in dataset : 22770
Number of RDD partitions : 2


# Phase 2: Data Cleaning
Using RDD transformations **only** to:
1. Fix malformed rows (wrong number of columns)
2. Remove rows with missing/empty values
3. Remove duplicate records with `distinct()`
4. Trim leading/trailing whitespace from all fields
5. Convert numeric columns to proper Python types

In [8]:
# A valid row in Salary.csv has exactly 8 columns:
# Rating, Company Name, Job Title, Salary, Salaries Reported, Location, Employment Status, Job Roles

def is_valid_row(cols):
    """Return True only if the row has exactly 8 non-empty columns
       and the numeric fields (Rating, Salary, Salaries Reported) are parseable."""
    # Check column count — fixes malformed/corrupted rows
    if len(cols) != 8:
        return False
    # Check for missing (empty) values in any column
    for col in cols:
        if col.strip() == "":
            return False
    # Validate numeric fields can be converted
    try:
        float(cols[0])   # Rating
        float(cols[3])   # Salary
        int(cols[4])     # Salaries Reported
    except ValueError:
        return False
    return True

# Apply the validation filter — this removes malformed and incomplete rows
valid_rdd = split_rdd.filter(is_valid_row)

removed = split_rdd.count() - valid_rdd.count()
print(f"Rows removed (malformed / missing values): {removed}")
print(f"Valid rows remaining                      : {valid_rdd.count()}")

Rows removed (malformed / missing values): 229
Valid rows remaining                      : 22541


In [9]:
# Use map() to clean every field:
#   - strip() removes surrounding whitespace from every string column
#   - numeric columns are cast to float or int

def clean_types(cols):
    """Trim whitespace and cast each column to its proper Python type."""
    return (
        float(cols[0].strip()),        # Rating           → float
        cols[1].strip(),               # Company Name     → str
        cols[2].strip(),               # Job Title        → str
        float(cols[3].strip()),        # Salary           → float
        int(cols[4].strip()),          # Salaries Reported→ int
        cols[5].strip(),               # Location         → str
        cols[6].strip(),               # Employment Status→ str
        cols[7].strip()                # Job Roles        → str
    )

# Apply the type-conversion map transformation
typed_rdd = valid_rdd.map(clean_types)

print("Type conversion applied. Sample typed record:")
print(typed_rdd.first())

Type conversion applied. Sample typed record:
(3.8, 'Sasken', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')


In [10]:
# distinct() removes exact duplicate rows from the RDD
# This handles cases where the same record appears more than once
cleaned_rdd = typed_rdd.distinct()

duplicates_removed = typed_rdd.count() - cleaned_rdd.count()
print(f"Duplicate records removed: {duplicates_removed}")
print(f"Final clean record count : {cleaned_rdd.count()}")

print("\nCleaned data sample (first 5 records):")
for record in cleaned_rdd.take(5):
    print(" ", record)

Duplicate records removed: 0


Final clean record count : 22541

Cleaned data sample (first 5 records):
  (3.8, 'Sasken', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (4.5, 'Advanced Millennium Technologies', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (4.2, 'Freelancer', 'Android Developer', 100000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (3.1, 'Samsung R&D Institute India - Bangalore', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (3.6, 'Craft Silicon', 'Android Developer', 300000.0, 3, 'Bangalore', 'Full Time', 'Android')


# Phase 3: Core RDD Transformations
Demonstrating **six** distinct RDD transformations, each in its own cell.

| # | Transformation | Purpose |
|---|---------------|---------|
| 1 | `flatMap()` | Flatten job title words |
| 2 | `filter()` | Keep only Full-Time jobs |
| 3 | `map()` | Extract company → salary pairs |
| 4 | `mapPartitions()` | Count records per partition |
| 5 | `sample()` | Draw a random data subset |
| 6 | `union()` | Merge two filtered RDDs |

In [11]:
# ── Transformation 1: flatMap() ──────────────────────────────────────────
# flatMap() applies a function that returns an iterable for each element,
# then flattens all those iterables into a single RDD.
#
# Purpose: Extract every individual word from the 'Job Title' column
#          to measure vocabulary / keyword frequency across all listings.

title_words_rdd = cleaned_rdd.flatMap(lambda row: row[2].split(" "))

print("flatMap → Individual words from 'Job Title'")
print("Sample words:", title_words_rdd.take(10))
print("Total word tokens:", title_words_rdd.count())

flatMap → Individual words from 'Job Title'
Sample words: ['Android', 'Developer', 'Android', 'Developer', 'Android', 'Developer', 'Android', 'Developer', 'Android', 'Developer']
Total word tokens: 73461


In [12]:
# ── Transformation 2: filter() ───────────────────────────────────────────
# filter() keeps only the elements for which the predicate returns True.
#
# Purpose: Isolate records where Employment Status is 'Full Time'
#          to analyze the full-time job market separately.

full_time_rdd = cleaned_rdd.filter(lambda row: row[6] == "Full Time")

total     = cleaned_rdd.count()
full_time = full_time_rdd.count()

print("filter → 'Full Time' employment records only")
print(f"Total records   : {total}")
print(f"Full-time records: {full_time}")
print(f"Percentage       : {full_time/total*100:.1f}%")

filter → 'Full Time' employment records only
Total records   : 22541
Full-time records: 19912
Percentage       : 88.3%


In [13]:
# ── Transformation 3: map() ──────────────────────────────────────────────
# map() applies a one-to-one transformation to each element, returning
# a new RDD of the same length.
#
# Purpose: Project the RDD down to (Company Name, Salary) pairs
#          to prepare for salary analysis per company.

company_salary_rdd = cleaned_rdd.map(lambda row: (row[1], row[3]))

print("map → (Company Name, Salary) pairs")
print("Sample pairs:")
for pair in company_salary_rdd.take(5):
    print(f"  {pair[0]!r:40s} → ₹{pair[1]:,.0f}")

map → (Company Name, Salary) pairs
Sample pairs:
  'Sasken'                                 → ₹400,000
  'Advanced Millennium Technologies'       → ₹400,000
  'Freelancer'                             → ₹100,000
  'Samsung R&D Institute India - Bangalore' → ₹400,000
  'Craft Silicon'                          → ₹300,000


In [14]:
# ── Transformation 4: mapPartitions() ────────────────────────────────────
# mapPartitions() works like map() but operates on an entire partition at once
# (receives an iterator). This reduces per-record overhead for expensive
# initialisation tasks (e.g., DB connections, ML model loading).
#
# Purpose: Count the number of records in each partition to observe
#          how Spark distributes data across LOCAL workers.

def count_in_partition(iterator):
    """Yield the number of elements in this partition."""
    yield sum(1 for _ in iterator)

partition_counts = cleaned_rdd.mapPartitions(count_in_partition)

counts = partition_counts.collect()
print("mapPartitions → Record count per partition")
for idx, cnt in enumerate(counts):
    print(f"  Partition {idx}: {cnt} records")

mapPartitions → Record count per partition
  Partition 0: 11083 records
  Partition 1: 11458 records


In [15]:
# ── Transformation 5: sample() ───────────────────────────────────────────
# sample() returns an approximate random subset of the RDD.
#   withReplacement=False → each element selected at most once
#   fraction             → approximate proportion to keep
#   seed                 → ensures reproducibility
#
# Purpose: Draw a 5% random sample for quick exploratory analysis
#          without processing the entire dataset.

sampled_rdd = cleaned_rdd.sample(withReplacement=False, fraction=0.05, seed=42)

print("sample → 5% random subset (seed=42)")
print(f"Original dataset size : {cleaned_rdd.count()}")
print(f"Sampled dataset size  : {sampled_rdd.count()}")
print("First 3 sampled records:")
for r in sampled_rdd.take(3):
    print(" ", r)

sample → 5% random subset (seed=42)


Original dataset size : 22541
Sampled dataset size  : 1061
First 3 sampled records:
  (3.8, 'Sasken', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (3.1, 'Samsung R&D Institute India - Bangalore', 'Android Developer', 400000.0, 3, 'Bangalore', 'Full Time', 'Android')
  (4.1, 'Fresher', 'Android Developer', 408000.0, 2, 'Bangalore', 'Full Time', 'Android')


In [16]:
# ── Transformation 6: union() ────────────────────────────────────────────
# union() concatenates two RDDs into one (does not deduplicate).
# Both RDDs must have compatible element types.
#
# Purpose: Combine 'Full Time' and 'Intern' employment records into a
#          single RDD to analyse both groups together while excluding
#          'Contractor' and other statuses.

intern_rdd   = cleaned_rdd.filter(lambda row: row[6] == "Intern")
combined_rdd = full_time_rdd.union(intern_rdd)

print("union → Full Time ∪ Intern records")
print(f"Full Time count : {full_time_rdd.count()}")
print(f"Intern count    : {intern_rdd.count()}")
print(f"Combined count  : {combined_rdd.count()}")

union → Full Time ∪ Intern records
Full Time count : 19912


Intern count    : 2056


Combined count  : 21968


# Phase 4: Key-Value Operations
All operations use the key-value pair `(Location, Salary)`.

| # | Operation | Purpose |
|---|-----------|---------|
| 1 | `reduceByKey()` | Total salary per location |
| 2 | `groupByKey()` | All salaries grouped by location |
| 3 | `countByKey()` | Job count per location |
| 4 | `sortByKey()` | Alphabetical location ordering |
| 5 | `aggregateByKey()` | Average salary per location |
| 6 | `combineByKey()` | Average salary (alternative approach) |

In [17]:
# ── Setup: Build the (Location, Salary) key-value RDD ────────────────────
# All Phase 4 operations build on this base pair.
# Key   = Location (str)
# Value = Salary   (float)

loc_salary_kv = cleaned_rdd.map(lambda row: (row[5], row[3]))

print("Key-Value RDD → (Location, Salary)")
print("Sample pairs:")
for pair in loc_salary_kv.take(5):
    print(f"  {pair}")

Key-Value RDD → (Location, Salary)
Sample pairs:
  ('Bangalore', 400000.0)
  ('Bangalore', 400000.0)
  ('Bangalore', 100000.0)
  ('Bangalore', 400000.0)
  ('Bangalore', 300000.0)


In [18]:
# ── Operation 1: reduceByKey() ───────────────────────────────────────────
# reduceByKey() merges values for identical keys using an associative,
# commutative function. It performs partial aggregation locally (combiner)
# before shuffling, making it more efficient than groupByKey() + reduce.
#
# Purpose: Calculate the TOTAL salary budget per location.

total_salary_per_loc = loc_salary_kv.reduceByKey(lambda a, b: a + b)

print("reduceByKey → Total salary per location (sample):")
for loc, total in total_salary_per_loc.take(5):
    print(f"  {loc:30s} → ₹{total:,.0f}")

reduceByKey → Total salary per location (sample):
  Hyderabad                      → ₹3,012,845,856
  New Delhi                      → ₹2,706,304,128
  Pune                           → ₹1,457,068,448
  Jaipur                         → ₹50,960,000
  Kerala                         → ₹59,786,368


In [19]:
# ── Operation 2: groupByKey() ────────────────────────────────────────────
# groupByKey() collects all values for each key into a ResultIterable.
# NOTE: This shuffles all values across the network — prefer reduceByKey()
#       or aggregateByKey() when you only need an aggregated result.
#
# Purpose: Inspect the full list of individual salaries per location,
#          which is needed when the list itself matters (not just a stat).

grouped_salaries = loc_salary_kv.groupByKey().mapValues(list)

print("groupByKey → All salaries grouped per location (2 sample locations):")
for loc, sals in grouped_salaries.take(2):
    print(f"  {loc}: {len(sals)} entries | min={min(sals):,.0f} | max={max(sals):,.0f}")

groupByKey → All salaries grouped per location (2 sample locations):
  Hyderabad: 4443 entries | min=8,448 | max=9,700,000
  New Delhi: 4112 entries | min=12,000 | max=90,000,000


In [20]:
# ── Operation 3: countByKey() ────────────────────────────────────────────
# countByKey() is an ACTION that returns a dict of {key: count}.
# It collects results to the driver, so only use it when the number
# of unique keys is small enough to fit in driver memory.
#
# Purpose: Count how many job listings exist per location.

loc_counts = loc_salary_kv.countByKey()

print("countByKey → Job listing count per location:")
# Sort by count descending for readability
for loc, cnt in sorted(loc_counts.items(), key=lambda x: -x[1])[:8]:
    print(f"  {loc:30s}: {cnt} listings")

countByKey → Job listing count per location:
  Bangalore                     : 8175 listings
  Hyderabad                     : 4443 listings
  New Delhi                     : 4112 listings
  Chennai                       : 2437 listings
  Pune                          : 2115 listings
  Mumbai                        : 743 listings
  Kolkata                       : 176 listings
  Madhya Pradesh                : 151 listings


In [21]:
# ── Operation 4: sortByKey() ─────────────────────────────────────────────
# sortByKey() sorts the RDD by key in ascending (default) or descending order.
# This triggers a full shuffle.
#
# Purpose: Present total-salary results in alphabetical order by location
#          for consistent, readable reporting.

sorted_locs = total_salary_per_loc.sortByKey(ascending=True)

print("sortByKey → Locations sorted alphabetically with their total salary:")
for loc, total in sorted_locs.take(8):
    print(f"  {loc:30s} → ₹{total:,.0f}")

sortByKey → Locations sorted alphabetically with their total salary:
  Bangalore                      → ₹5,987,612,928
  Chennai                        → ₹1,425,999,648
  Hyderabad                      → ₹3,012,845,856
  Jaipur                         → ₹50,960,000
  Kerala                         → ₹59,786,368
  Kolkata                        → ₹126,118,752
  Madhya Pradesh                 → ₹102,326,496
  Mumbai                         → ₹712,524,096


In [22]:
# ── Operation 5: aggregateByKey() ────────────────────────────────────────
# aggregateByKey() allows the accumulator type to differ from the value type.
# It needs three arguments:
#   zeroValue  — initial accumulator per partition
#   seqOp      — how to merge a value into an accumulator (within partition)
#   combOp     — how to merge two accumulators (across partitions)
#
# Purpose: Compute AVERAGE salary per location using a (sum, count) tuple.

zero_val = (0.0, 0)  # (running_sum, running_count)

def seq_op(acc, value):
    """Merge one salary value into the partition accumulator."""
    return (acc[0] + value, acc[1] + 1)

def comb_op(acc1, acc2):
    """Merge two partition accumulators during the shuffle."""
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

avg_salary_rdd = (
    loc_salary_kv
    .aggregateByKey(zero_val, seq_op, comb_op)
    .mapValues(lambda x: round(x[0] / x[1], 2))
)

print("aggregateByKey → Average salary per location (sorted, sample):")
for loc, avg in avg_salary_rdd.sortByKey().take(5):
    print(f"  {loc:30s} → ₹{avg:,.2f}")

aggregateByKey → Average salary per location (sorted, sample):


  Bangalore                      → ₹732,429.72
  Chennai                        → ₹585,145.53
  Hyderabad                      → ₹678,110.70
  Jaipur                         → ₹629,135.80
  Kerala                         → ₹553,577.48


In [23]:
# ── Operation 6: combineByKey() ──────────────────────────────────────────
# combineByKey() is the most general key-value aggregation primitive.
# It requires three functions:
#   createCombiner — initialise an accumulator for the first value of a key
#   mergeValue     — add a new value to an existing accumulator
#   mergeCombiners — combine two accumulators (across partitions)
#
# Purpose: Compute AVERAGE salary per location (alternative to aggregateByKey).

def create_combiner(val):
    """First value for a key: start a (sum, count) tuple."""
    return (val, 1)

def merge_value(acc, val):
    """Add another value to the running total."""
    return (acc[0] + val, acc[1] + 1)

def merge_combiners(acc1, acc2):
    """Combine two partition-level (sum, count) tuples."""
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

avg_combine_rdd = (
    loc_salary_kv
    .combineByKey(create_combiner, merge_value, merge_combiners)
    .mapValues(lambda x: round(x[0] / x[1], 2))
)

print("combineByKey → Average salary per location (sorted, sample):")
for loc, avg in avg_combine_rdd.sortByKey().take(5):
    print(f"  {loc:30s} → ₹{avg:,.2f}")

combineByKey → Average salary per location (sorted, sample):


  Bangalore                      → ₹732,429.72
  Chennai                        → ₹585,145.53
  Hyderabad                      → ₹678,110.70
  Jaipur                         → ₹629,135.80
  Kerala                         → ₹553,577.48


# Phase 5: Partitioning & Performance
This phase covers:
1. Inspecting the initial partition count
2. Benchmarking execution time **without** caching
3. Applying `cache()` and benchmarking **with** caching
4. Changing partition count with `repartition()` and `coalesce()`
5. Discussing the performance observations

In [24]:
# ── Step 1: Inspect initial partition count ───────────────────────────────
# Each partition is an independent chunk of data that Spark can process
# in parallel on a separate CPU core or machine.
# The default number of partitions is typically tied to the number of
# input file splits (128 MB per split for HDFS, or CPU cores for local mode).

initial_partitions = cleaned_rdd.getNumPartitions()
print(f"Initial partition count of cleaned_rdd: {initial_partitions}")
print(f"Available CPU cores (local[*])         : reported above in Spark UI")

Initial partition count of cleaned_rdd: 2
Available CPU cores (local[*])         : reported above in Spark UI


In [25]:
import time

# ── Step 2: Measure execution time WITHOUT caching ────────────────────────
# Without cache(), every ACTION re-reads the source file and recomputes
# the entire transformation chain from scratch (Spark's lazy evaluation).

def heavy_computation(rdd):
    """Simulate a multi-step analytical pipeline."""
    return (
        rdd
        .map(lambda x: (x[5], x[3] * 1.10))      # apply 10% salary uplift
        .reduceByKey(lambda a, b: a + b)           # total per location
        .sortByKey()                               # order alphabetically
        .collect()                                 # trigger execution
    )

start = time.time()
result_no_cache = heavy_computation(cleaned_rdd)
time_no_cache   = time.time() - start

print(f"Execution time WITHOUT caching: {time_no_cache:.4f} s")
print(f"(Result sample: {result_no_cache[:3]})")

Execution time WITHOUT caching: 1.8293 s
(Result sample: [('Bangalore', 6586374220.8), ('Chennai', 1568599612.8000002), ('Hyderabad', 3314130441.6)])


In [26]:
# ── Step 3: Cache the RDD and re-measure ─────────────────────────────────
# cache() marks the RDD to be stored in memory after its first computation.
# persist() is the generalised form — cache() = persist(MEMORY_ONLY).
# The first action after cache() still computes the full lineage,
# but subsequent actions reuse the in-memory copy.

cleaned_rdd.cache()          # mark for caching
cleaned_rdd.count()          # materialise the cache (first pass)

start = time.time()
result_cached = heavy_computation(cleaned_rdd)
time_cached   = time.time() - start

print(f"Execution time WITH caching : {time_cached:.4f} s")
print(f"Execution time WITHOUT cache: {time_no_cache:.4f} s")
speedup = time_no_cache / time_cached if time_cached > 0 else float('inf')
print(f"Speed-up factor             : {speedup:.2f}x")

Execution time WITH caching : 1.7876 s
Execution time WITHOUT cache: 1.8293 s
Speed-up factor             : 1.02x


In [27]:
# ── Step 4: Adjust partition count ───────────────────────────────────────
# repartition(n):
#   - Increases OR decreases partitions via a full shuffle.
#   - Use when you need evenly distributed partitions (e.g., before a join
#     or to parallelise write operations more evenly).
#
# coalesce(n):
#   - Only DECREASES partitions, avoids a full shuffle by combining
#     adjacent partitions on the same worker.
#   - More efficient than repartition() when reducing partition count.

print(f"Partition count BEFORE adjustment : {cleaned_rdd.getNumPartitions()}")

repartitioned_rdd = cleaned_rdd.repartition(4)
print(f"After repartition(4)              : {repartitioned_rdd.getNumPartitions()} partitions")

coalesced_rdd = repartitioned_rdd.coalesce(2)
print(f"After coalesce(2)                 : {coalesced_rdd.getNumPartitions()} partitions")

Partition count BEFORE adjustment : 2
After repartition(4)              : 4 partitions
After coalesce(2)                 : 2 partitions


In [28]:
# ── Step 5: Performance Observations ─────────────────────────────────────
observations = """
PERFORMANCE SUMMARY
===================
1. Caching (cache / persist):
   - Without caching: every action recomputes the full transformation
     lineage from the raw file → slower for repeated operations.
   - With caching: data is stored in JVM memory after the first pass,
     so subsequent actions skip re-reading/re-computing → faster.
   - Best used for RDDs that are accessed multiple times (e.g., in
     iterative ML algorithms or multi-step reports).

2. repartition():
   - Triggers a full shuffle across the network to create evenly-sized
     partitions.
   - Useful when scaling UP partitions to increase parallelism.
   - Higher cost upfront, but pays off for large downstream operations.

3. coalesce():
   - Combines partitions on the same node WITHOUT a full shuffle.
   - Only reduces the partition count; cannot increase it.
   - Preferred when reducing partitions (e.g., before saving output)
     because it avoids expensive network I/O.
"""
print(observations)


PERFORMANCE SUMMARY
1. Caching (cache / persist):
   - Without caching: every action recomputes the full transformation
     lineage from the raw file → slower for repeated operations.
   - With caching: data is stored in JVM memory after the first pass,
     so subsequent actions skip re-reading/re-computing → faster.
   - Best used for RDDs that are accessed multiple times (e.g., in
     iterative ML algorithms or multi-step reports).

2. repartition():
   - Triggers a full shuffle across the network to create evenly-sized
     partitions.
   - Useful when scaling UP partitions to increase parallelism.
   - Higher cost upfront, but pays off for large downstream operations.

3. coalesce():
   - Combines partitions on the same node WITHOUT a full shuffle.
   - Only reduces the partition count; cannot increase it.
   - Preferred when reducing partitions (e.g., before saving output)
     because it avoids expensive network I/O.



In [29]:
# ── Cleanup: Stop Spark session ──────────────────────────────────────────
# Always stop the SparkContext / SparkSession when finished to release
# cluster/local resources (threads, memory, port bindings).

sc.stop()
spark.stop()
print("Spark session stopped.")

Spark session stopped.
